# 🌍 Earth Field Analysis
## Layer 5 – Global Electric Circuit

| Layer | Name | Status |
|-------|------|--------|
| 0 | External Cosmic Drivers | ✅ `layer0_state.json` |
| 1 | Planetary Body | ✅ `layer1_state.json` |
| 2 | Surface / Oceans / Land | ✅ `layer2_state.json` |
| 3 | Atmosphere / Weather / Thunderstorms | ✅ `layer3_state.json` |
| 4 | Ionosphere | ✅ `layer4_state.json` |
| **5** | **Global Electric Circuit** | **← this layer** |
| 6 | Resonance Field / Schumann | ⬜ |
| 7 | Earth Field State Engine | ⬜ |
| 8 | Research / Hypotheses | ⬜ |

> **Core idea:** Thunderstorms charge the ionosphere → potential difference arises → vertical currents flow → global electrical cycle
>
> **Significance:** Layer 5 is the system architecture in which Schumann resonance is embedded.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, re, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

print(f'Analysis date: {datetime.date.today()}')

# Load all previous layers
context = {}
for n in [0, 1, 2, 3, 4]:
    try:
        with open(f'layer{n}_test_state.json', encoding='utf-8') as f:
            context[n] = json.load(f)
        print(f'  Layer {n}: {context[n]["level"].upper():8}  Score={context[n]["score"]}')
    except FileNotFoundError:
        print(f'  Layer {n}: not found')
        context[n] = None

layer0 = context.get(0)
layer1 = context.get(1)
layer2 = context.get(2)
layer3 = context.get(3)
layer4 = context.get(4)

# Extract relevant upstream values
# Layer 3: thunderstorm activity (generator side)
l3_thunder_score  = layer3.get('components',{}).get('Thunderstorm Activity',{}).get('score') if layer3 else None
l3_schumann_pot   = layer3.get('raw_values',{}).get('schumann_potential') if layer3 else None
l3_cape           = layer3.get('raw_values',{}).get('CAPE_mean_Jkg') if layer3 else None
l3_n_thunder      = layer3.get('raw_values',{}).get('thunder_points_WMO') if layer3 else None
l3_thunder_flag   = layer3.get('flags',{}).get('active_thunderstorms') if layer3 else False

# Layer 4: Ionospheren-Leitfähigkeit (obere Elektrode)
l4_ioniz          = layer4.get('components',{}).get('Ionization Level (F10.7)',{}).get('score') if layer4 else None
l4_disturb        = layer4.get('components',{}).get('Disturbance Level (Kp)',{}).get('score') if layer4 else None
l4_cavity_h       = layer4.get('resonance_system',{}).get('cavity_height_km') if layer4 else 80.0
l4_kp             = layer4.get('raw_values',{}).get('Kp_current') if layer4 else None
l4_f107           = layer4.get('raw_values',{}).get('F10.7_sfu') if layer4 else None
l4_sr1_hz         = layer4.get('resonance_system',{}).get('schumann_frequencies_Hz',{}).get('SR_1') if layer4 else None

# Layer 1: ground/ocean conductivity (lower electrode)
l1_leitf          = layer1.get('baseline_properties',{}).get('ocean_conductivity_Sm') if layer1 else 3.2
l1_ocean_frac     = layer1.get('baseline_properties',{}).get('ocean_surface_fraction') if layer1 else 0.71

print(f'\n  L3: thunder_score={l3_thunder_score}  CAPE={l3_cape}  thunder_pts={l3_n_thunder}')
print(f'  L4: ioniz={l4_ioniz}  Kp={l4_kp}  cavity_h={l4_cavity_h} km  SR-1={l4_sr1_hz} Hz')
print(f'  L1: ocean_cond={l1_leitf} S/m  ocean_frac={l1_ocean_frac}')

Analysis date: 2026-05-14
  Layer 0: CALM      Score=0.1947
  Layer 1: CALM      Score=0.2455
  Layer 2: ACTIVE    Score=0.6036
  Layer 3: QUIET     Score=0.1432
  Layer 4: MODERATE  Score=0.326

  L3: thunder_score=0.1811  CAPE=640.0  thunder_pts=0
  L4: ioniz=0.316  Kp=1.0  cavity_h=69.2 km  SR-1=10.4775 Hz
  L1: ocean_cond=3.2 S/m  ocean_frac=0.71


---
## 1. GEC Architecture: System Structure

```
┌─────────────────────────────────────────────────────┐
│  IONOSPHERE  (~85–100 km)   upper electrode (+300 kV) │  ← Layer 4
│  High conductivity, equipotential surface             │
├──────────────────────────────────────────────────────┤
│  FAIR-WEATHER ZONE           vertical return current  │
│  2 pA/m²  ~1 kV/m           ~1000 A global          │
├──────────────────────────────────────────────────────┤
│  THUNDERSTORM ZONES (L3)     charging / generator     │  ← Layer 3
│  ~45 lightning/s  ~1 GW     ~2000 simultaneous        │
├──────────────────────────────────────────────────────┤
│  EARTH SURFACE / OCEANS     lower electrode (0 V)     │  ← Layer 1/2
│  σ_ocean = 3.2 S/m          σ_rock ≈ 10⁻⁴ S/m         │
└─────────────────────────────────────────────────────┘
```

**Reference values (quiet state):**

| Quantity | Value | Unit |
|-------|------|---------|
| Ionospherenpotential | ~300 | kV |
| Global current | ~1000 | A |
| Fair-weather field strength | ~100–150 | V/m |
| Fair-weather current density | ~2 | pA/m² |
| Simultaneous thunderstorms | ~2000 | – |
| Lightning global | ~45 | /s |

In [2]:
# ============================================================
# GEC STRUCTURE DIAGRAM (simplified, not to scale)
# ============================================================

fig = go.Figure()

# Schichten als Rechtecke
layers_gec = [
    {
        'y0': 85, 'y1': 100, 'color': '#2A2470',
        'title': 'Ionosphere  (~+250–300 kV)',
        'sub':   'upper electrode / equipotential surface'
    },
    {
        'y0': 15, 'y1': 85, 'color': '#0d0d1a',
        'title': 'Fair-Weather Zone',
        'sub':   'downward return current ~2 pA/m²  |  E-field near ground ~100 V/m'
    },
    {
        'y0': 5, 'y1': 15, 'color': '#5A1A00',
        'title': 'Troposphere / Thunderstorm Generator Zone',
        'sub':   'Thunderstorms as local generators of the Global Electric Circuit'
    },
    {
        'y0': 0, 'y1': 5, 'color': '#1a1a0a',
        'title': 'Earth Surface / Oceans / Land  (0 V)',
        'sub':   'lower electrode / conductive boundary  |  ocean conductivity ~3.2 S/m'
    },
]

for l in layers_gec:
    fig.add_shape(type='rect', x0=0.08, x1=0.92,
                  y0=l['y0'], y1=l['y1'],
                  fillcolor=l['color'], opacity=0.85,
                  line=dict(color='#334', width=1))
    mid = (l['y0'] + l['y1']) / 2
    fig.add_annotation(
        x=0.50, y=mid + 1.5,
        text=f"<b>{l['title']}</b>",
        showarrow=False, xanchor='center',
        font=dict(size=10, color='white')
    )
    fig.add_annotation(
        x=0.50, y=mid - 2.5,
        text=l['sub'],
        showarrow=False, xanchor='center',
        font=dict(size=8.5, color='#AAAACC')
    )

# Aufwärtsstrom-Pfeile (gelb, Gewitter → Ionosphere)
for x_pos in [0.20, 0.50, 0.80]:
    fig.add_annotation(
        x=x_pos, y=84, ax=x_pos, ay=16,
        xref='x', yref='y', axref='x', ayref='y',
        arrowhead=3, arrowsize=1.3, arrowwidth=2.5,
        arrowcolor='#F2A623', showarrow=True, text=''
    )

# Rückstrom-Pfeile (blau, Ionosphere → Boden)
for x_pos in [0.29, 0.59, 0.89]:
    fig.add_annotation(
        x=x_pos, y=16, ax=x_pos, ay=84,
        xref='x', yref='y', axref='x', ayref='y',
        arrowhead=3, arrowsize=1.2, arrowwidth=2,
        arrowcolor='#378ADD', showarrow=True, text=''
    )

# Arrow legend (top right)
fig.add_annotation(x=0.97, y=96,
    text='<b>⬆</b> Thunderstorm charging current',
    showarrow=False, xanchor='right',
    font=dict(size=9, color='#F2A623'))
fig.add_annotation(x=0.97, y=91,
    text='<b>⬇</b> Fair-weather return current',
    showarrow=False, xanchor='right',
    font=dict(size=9, color='#378ADD'))

# Layer-3 status line (bottom)
n_pts   = l3_n_thunder if l3_n_thunder is not None else 0
cape_str = f'CAPE mean {l3_cape:.0f} J/kg' if l3_cape else 'CAPE n/a'
status_color = '#E85D24' if l3_thunder_flag else '#2ecc71'
status_level = layer3['level'].upper() if layer3 else 'n/a'
fig.add_annotation(
    x=0.50, y=-9,
    text=f'Layer 3 Status: {status_level}  |  {n_pts} WMO thunderstorm points  |  {cape_str}',
    showarrow=False, xanchor='center',
    font=dict(size=9.5, color=status_color)
)

fig.update_layout(
    title=dict(
        text='Global Electric Circuit – Simplified structure diagram (not to scale)',
        font=dict(size=14)
    ),
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0, 1]),
    yaxis=dict(title='Altitude [km]', range=[-14, 108],
               gridcolor='#222244', tickcolor='#888'),
    height=500, plot_bgcolor='#0d0d1a', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=20, t=60, b=30)
)
fig.show()


---
## 2. Fetch Real-Time Data

In [3]:
# ============================================================
# REAL-TIME DATA – THREE SOURCES
# 1) NOAA SWPC  – Kp + IMF (ionospheric conductivity)
# 2) NOAA SWPC  – X-Ray (solar ionization → conductivity)
# 3) USGS       – seismics as noise in ground current circuit
# ============================================================

raw = {}

# --- 1a) Kp ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/planetary_k_index_1m.json', timeout=12)
    r.raise_for_status(); raw['kp'] = r.json()
    print(f'  OK Kp                  {len(raw["kp"]):>5} entries')
except Exception as e:
    print(f'  ERR Kp                 {str(e)[:60]}')
    raw['kp'] = None

# --- 1b) IMF Bz ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json', timeout=12)
    r.raise_for_status(); raw['imf'] = r.json()
    print(f'  OK IMF Bz              {len(raw["imf"]):>5} entries')
except Exception as e:
    print(f'  ERR IMF Bz             {str(e)[:60]}')
    raw['imf'] = None

# --- 1c) Solar wind (speed / density) ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json', timeout=12)
    r.raise_for_status(); raw['sw'] = r.json()
    print(f'  OK Solar wind          {len(raw["sw"]):>5} entries')
except Exception as e:
    print(f'  ERR Solarwind          {str(e)[:60]}')
    raw['sw'] = None

# --- 2) X-Ray (D-Layer ionization level) ---
try:
    r = requests.get('https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json', timeout=12)
    r.raise_for_status(); raw['xray'] = r.json()
    print(f'  OK X-Ray               {len(raw["xray"]):>5} entries')
except Exception as e:
    print(f'  ERR X-Ray              {str(e)[:60]}')
    raw['xray'] = None

# --- 3) F10.7 ---
try:
    r = requests.get(
        'https://services.swpc.noaa.gov/json/solar-cycle/observed-solar-cycle-indices.json',
        timeout=12)
    r.raise_for_status(); raw['f107'] = r.json()
    last = raw['f107'][-1]
    print(f'  OK F10.7               {last.get("f10.7","–")} sfu')
except Exception as e:
    print(f'  ERR F10.7              {str(e)[:60]}')
    raw['f107'] = None

print('\nData fetch complete')

  OK Kp                    358 entries
  OK IMF Bz               2839 entries


  OK Solar wind           2830 entries
  OK X-Ray                2874 entries
  OK F10.7               120.01 sfu

Data fetch complete


In [4]:
# ============================================================
# PROCESS DATA
# ============================================================

def kp_strip(s):
    m = re.match(r'([0-9]+(?:\.[0-9]*)?)', str(s))
    return float(m.group(1)) if m else None

# --- Kp ---
kp_now = None; kp_max_24h = None
if raw['kp']:
    df_kp = pd.DataFrame(raw['kp'])
    tc = next((c for c in df_kp.columns if 'time' in c.lower()), df_kp.columns[0])
    kc = 'kp' if 'kp' in df_kp.columns else next(
        (c for c in df_kp.columns if 'kp' in c.lower() and c != tc), df_kp.columns[1])
    df_kp['time'] = pd.to_datetime(df_kp[tc])
    df_kp['kp']   = df_kp[kc].apply(kp_strip)
    df_kp = df_kp[df_kp['kp'] >= 0].dropna(subset=['kp']).sort_values('time').tail(1440)
    if not df_kp.empty:
        kp_now = float(df_kp['kp'].iloc[-1])
        kp_max_24h = float(df_kp['kp'].max())
        print(f'Kp: {kp_now:.1f}  max(24h): {kp_max_24h:.1f}')

# --- IMF Bz ---
bz_now = None
if raw['imf']:
    df_imf = pd.DataFrame(raw['imf'])
    if 'bz_gsm' in df_imf.columns:
        df_imf['bz_gsm'] = pd.to_numeric(df_imf['bz_gsm'], errors='coerce')
        s = df_imf['bz_gsm'].dropna()
        bz_now = float(s.iloc[-1]) if not s.empty else None
        print(f'IMF Bz: {bz_now:+.1f} nT')

# --- Solar wind ---
sw_speed = None; sw_density = None
if raw['sw']:
    df_sw = pd.DataFrame(raw['sw'])
    for col in ['speed', 'density']:
        if col in df_sw.columns:
            df_sw[col] = pd.to_numeric(df_sw[col], errors='coerce')
    if 'speed' in df_sw.columns:
        s = df_sw['speed'].dropna()
        sw_speed = float(s.iloc[-1]) if not s.empty else None
    if 'density' in df_sw.columns:
        s = df_sw['density'].dropna()
        sw_density = float(s.iloc[-1]) if not s.empty else None
    print(f'Solar wind: v={sw_speed:.0f} km/s  n={sw_density:.1f} cm⁻³' if sw_speed else 'SW: n/a')

# --- X-Ray ---
xray_now = None; xray_class = None
if raw['xray']:
    df_xr = pd.DataFrame(raw['xray'])
    tc = next((c for c in df_xr.columns if 'time' in c.lower()), df_xr.columns[0])
    fc = next((c for c in df_xr.columns
               if any(k in c.lower() for k in ['flux','long']) and c != tc), df_xr.columns[1])
    df_xr['flux'] = pd.to_numeric(df_xr[fc], errors='coerce')
    s = df_xr['flux'].dropna()
    if not s.empty:
        xray_now = float(s.iloc[-1])
        xray_class = ('X' if xray_now>=1e-4 else 'M' if xray_now>=1e-5
                      else 'C' if xray_now>=1e-6 else 'B' if xray_now>=1e-7 else 'A')
        print(f'X-Ray: {xray_class}-class ({xray_now:.2e} W/m²)')

# --- F10.7 ---
f107_now = None
if raw['f107']:
    try:
        vals = [float(e.get('f10.7',0)) for e in raw['f107'] if e.get('f10.7')]
        f107_now = vals[-1] if vals else None
        print(f'F10.7: {f107_now:.1f} sfu')
    except: pass

print('\nData processing complete')

Kp: 1.0  max(24h): 3.0
IMF Bz: +3.7 nT
SW: n/a
X-Ray: C-class (1.35e-06 W/m²)
F10.7: 120.0 sfu

Data processing complete


---
## 3. GEC Modelling: Current, Potential, Conductivity

In [5]:
# ============================================================
# GEC REFERENCE VALUES (Carnegie curve / literature)
# ============================================================

GEC_REF = {
    'V_iono_kV':        300.0,   # Ionospherenpotential [kV]
    'I_total_A':        1000.0,  # Global total current [A]
    'E_field_Vm':       130.0,   # Fair-weather E-field [V/m]
    'J_fair_pAm2':      2.0,     # Fair-weather current density [pA/m²]
    'n_thunderstorms':  2000,    # Simultaneous thunderstorms (global)
    'lightning_per_s':  45.0,    # Lightning per second (global)
    'power_W':          1e9,     # Generator power [W]
    'tau_relax_min':    10.0,    # Relaxation time [min]
}

# ============================================================
# GEC DYNAMIC MODELLING
# How do current Layer-3/4 states modify the GEC?
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return round(max(0.0, min(1.0, (v - lo) / (hi - lo))), 4)

# --- GENERATOR SIDE (Layer 3) ---
# Thunderstorm current modulator: relative to 2000 thunderstorms
# We have 6 measurement points → scale up to global
# Proxy: CAPE mean as generator strength
cape_ref   = 800.0   # J/kg – moderate global mean
cape_now   = l3_cape if l3_cape is not None else cape_ref
gen_factor = max(0.3, min(2.0, cape_now / cape_ref))  # 0.3x – 2.0x

# Thunderstorm score as direct generator proxy
thunder_factor = max(0.5, min(1.8, 1.0 + (l3_thunder_score or 0.15) * 2))

generator_strength = round((gen_factor * 0.5 + thunder_factor * 0.5), 3)

# --- CONDUCTIVITY SIDE (Layer 4 + Layer 1) ---
# Ionospheric conductivity: higher at high F10.7
f107_val = f107_now or l4_f107 or 120.0
iono_cond_factor = max(0.7, min(1.4, f107_val / 120.0))

# Ground conductivity: stable (71% oceans)
ground_cond = (l1_leitf or 3.2) * (l1_ocean_frac or 0.71)

# Kp modulator: Storm reduces ionospheric resistance → more power
kp_val = kp_now or l4_kp or 2.0
kp_cond_mod = 1.0 + max(0, kp_val - 3) * 0.05  # slight increase during storm

# --- GEC STATE CALCULATION ---
# Modulated ionosphere potential
V_iono_mod = round(GEC_REF['V_iono_kV'] * generator_strength * iono_cond_factor, 1)

# Modulated total current
I_total_mod = round(GEC_REF['I_total_A'] * generator_strength * iono_cond_factor * kp_cond_mod, 1)

# Fair-weather E-field
E_mod = round(GEC_REF['E_field_Vm'] * generator_strength * iono_cond_factor, 1)

# Fair-weather current density
J_mod = round(GEC_REF['J_fair_pAm2'] * generator_strength, 3)

# Relative deviation from reference (Carnegie)
delta_V_pct = round((V_iono_mod - GEC_REF['V_iono_kV']) / GEC_REF['V_iono_kV'] * 100, 1)
delta_I_pct = round((I_total_mod - GEC_REF['I_total_A']) / GEC_REF['I_total_A'] * 100, 1)

print('GEC STATE MODEL')
print('=' * 58)
print(f'  Generator strength:         {generator_strength:.3f}  (Ref: 1.0)')
print(f'  Iono. cond. factor:         {iono_cond_factor:.3f}  (F10.7={f107_val:.0f} sfu)')
print(f'  Kp modulator:               {kp_cond_mod:.3f}  (Kp={kp_val:.1f})')
print('-' * 58)
print(f'  Ionospherenpotential:        {V_iono_mod:>7.1f} kV   (Ref: {GEC_REF["V_iono_kV"]:.0f} kV   Δ {delta_V_pct:+.1f}%)')
print(f'  Global current:              {I_total_mod:>7.1f} A    (Ref: {GEC_REF["I_total_A"]:.0f} A    Δ {delta_I_pct:+.1f}%)')
print(f'  Fair-weather E-field:        {E_mod:>7.1f} V/m  (Ref: {GEC_REF["E_field_Vm"]:.0f} V/m)')
print(f'  Fair-weather current dens.:  {J_mod:>7.3f} pA/m²(Ref: {GEC_REF["J_fair_pAm2"]:.1f} pA/m²)')
print('=' * 58)

GEC STATE MODEL
  Generator strength:         1.081  (Ref: 1.0)
  Iono. cond. factor:         1.000  (F10.7=120 sfu)
  Kp modulator:               1.000  (Kp=1.0)
----------------------------------------------------------
  Ionospherenpotential:          324.3 kV   (Ref: 300 kV   Δ +8.1%)
  Global current:               1081.1 A    (Ref: 1000 A    Δ +8.1%)
  Fair-weather E-field:          140.5 V/m  (Ref: 130 V/m)
  Fair-weather current dens.:    2.162 pA/m²(Ref: 2.0 pA/m²)


---
## 4. Visualizations

In [6]:
# ============================================================
# GEC STATE vs. REFERENCE
# Note: raw values have different units (kV, A, V/m)
# Comparison shows relative deviation from Carnegie reference
# ============================================================

metrics = [
    ('Ionosph. Potential', 'kV',  GEC_REF['V_iono_kV'], V_iono_mod),
    ('Global Current',     'A',   GEC_REF['I_total_A'], I_total_mod),
    ('E-Field (Fair-W.)',  'V/m', GEC_REF['E_field_Vm'], E_mod),
]
labels_units = [f'{m[0]}\n[{m[1]}]' for m in metrics]

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Reference (Carnegie)',
    x=labels_units, y=[m[2] for m in metrics],
    marker_color='#5F5E5A', opacity=0.75
))
fig.add_trace(go.Bar(
    name='Current (modelled)',
    x=labels_units, y=[m[3] for m in metrics],
    marker_color='#F2A623', opacity=0.90,
    text=[f'<b>{m[3]:.1f} {m[1]}</b>' for m in metrics],
    textposition='outside',
    textfont=dict(color='white', size=11)
))

# Joint percentage value as annotation
delta_pct = round((V_iono_mod - GEC_REF['V_iono_kV']) / GEC_REF['V_iono_kV'] * 100, 1)
fig.add_annotation(
    x=1.0, y=max(GEC_REF['V_iono_kV'], V_iono_mod) * 0.95,
    text=f'All values {delta_pct:+.1f}% vs. reference<br>(shared generator factor)',
    showarrow=False, xanchor='center',
    font=dict(color='#F2A623', size=10),
    bgcolor='rgba(20,20,40,0.7)'
)

fig.update_layout(
    title=dict(
        text='GEC State: Current vs. Reference (Carnegie)  —  normalized comparison values, different units',
        font=dict(size=13)
    ),
    barmode='group',
    height=380,
    plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=50, r=50, t=65, b=60),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white', size=10)),
    yaxis=dict(gridcolor='#2a2a4a')
)
fig.show()

print(f'GEC deviation from Carnegie reference: {delta_pct:+.1f}%')
print(f'  (all values scale with the same generator factor {generator_strength:.3f})')


GEC deviation from Carnegie reference: +8.1%
  (all values scale with the same generator factor 1.081)


In [7]:
# ============================================================
# KP TIME SERIES + IMF BZ (GEC modulator last 24h)
# ============================================================

panels = []
if raw['kp']:    panels.append(('kp',     df_kp,  'Kp-Index (Ionospheric conductivity)', '#7F77DD', 'bar'))
if raw['imf'] and 'bz_gsm' in pd.DataFrame(raw['imf']).columns:
    df_imf2 = pd.DataFrame(raw['imf'])
    tc2 = next((c for c in df_imf2.columns if 'time' in c.lower()), df_imf2.columns[0])
    df_imf2['time']   = pd.to_datetime(df_imf2[tc2])
    df_imf2['bz_gsm'] = pd.to_numeric(df_imf2['bz_gsm'], errors='coerce')
    df_imf2 = df_imf2.dropna(subset=['bz_gsm']).sort_values('time').tail(1440)
    panels.append(('bz_gsm', df_imf2, 'IMF Bz [nT] (M-I Coupling)', '#534AB7', 'line'))

if panels:
    fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True,
                        subplot_titles=[p[2] for p in panels], vertical_spacing=0.12)
    for i, (col, df, title, color, kind) in enumerate(panels, 1):
        ds = df.dropna(subset=[col])
        if kind == 'bar':
            mc = ['#2ecc71' if v < 4 else '#f39c12' if v < 6 else '#e74c3c' for v in ds[col]]
            fig.add_trace(go.Bar(x=ds['time'], y=ds[col], marker_color=mc, opacity=0.8), row=i, col=1)
            fig.add_hline(y=5, line_dash='dot', line_color='#e74c3c', row=i, col=1)
            fig.update_yaxes(range=[0, 9], row=i, col=1)
        else:
            mc = ['#e74c3c' if v < -5 else '#2ecc71' if v > 5 else '#888780' for v in ds[col]]
            fig.add_trace(go.Bar(x=ds['time'], y=ds[col], marker_color=mc, opacity=0.75), row=i, col=1)
            fig.add_hline(y=0, line_color='gray', line_width=0.5, row=i, col=1)
            fig.add_hline(y=-5, line_dash='dot', line_color='#e74c3c',
                          annotation_text='GEC coupling active', row=i, col=1)

    fig.update_layout(
        title=dict(text='GEC Modulators – last 24h', font=dict(size=14)),
        height=180*len(panels)+100, showlegend=False,
        plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=55, b=40)
    )
    fig.show()

In [8]:
# ============================================================
# COUPLING MATRIX – How strongly does each layer influence the GEC?
# ============================================================

# Coupling strengths (0–1) from upstream layer scores
def safe(v, default=0.0):
    return float(v) if v is not None else default

coupling = {
    'L3 → GEC\n(Thunderstorm/Generator)':    safe(l3_thunder_score, 0.15),
    'L4 → GEC\n(Iono. Conductivity)':    safe(l4_ioniz, 0.3),
    'L1 → GEC\n(Ground Conductivity)':   0.71,   # stabil (Ozeananteil)
    'L0 → GEC\n(Solar/Kp Modulator)':    safe(layer0.get('score') if layer0 else None, 0.25),
    'L2 → GEC\n(Surface Coupling)':      safe(layer2.get('components',{}).get('Elec. Surface Coupling',{}).get('score') if layer2 else None, 0.3),
}

names  = list(coupling.keys())
values = list(coupling.values())
colors = ['#E85D24','#534AB7','#185FA5','#F2A623','#639922']

fig = go.Figure(go.Bar(
    x=values, y=names,
    orientation='h',
    marker_color=colors, opacity=0.85,
    text=[f'{v:.3f}' for v in values],
    textposition='outside',
    textfont=dict(color='white')
))
fig.add_vline(x=0.5, line_dash='dot', line_color='#e74c3c',
              annotation_text='Elevated Coupling', annotation_position='top')
fig.update_layout(
    title=dict(text='GEC Coupling Matrix: Upstream Layer Influence', font=dict(size=14)),
    xaxis=dict(title='Coupling strength [0–1]', range=[0, 1.15]),
    height=340, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=180, r=70, t=55, b=40)
)
fig.show()

---
## 5. State Assessment & Handoff to Layer 6

In [9]:
# ============================================================
# LAYER-5-SCORE
# ============================================================

# Components
# 1) Generator activity (Layer-3-driven)
gen_score = norm(generator_strength, 0.3, 2.0)
gen_src   = 'from_layer3' if l3_thunder_score is not None else 'estimated'

# 2) Ionospheric conductivity (Layer-4-driven)
iono_score = norm(iono_cond_factor, 0.7, 1.4)
iono_src   = 'from_layer4' if l4_ioniz is not None else 'estimated'

# 3) GEC potential deviation from reference
pot_score = norm(abs(delta_V_pct), 0, 30)  # 0 = at reference, 1 = 30% deviation
pot_src   = 'derived'

# 4) Geomagnetic disturbance (Kp) – modulates vertical currents
kp_score = norm(kp_val, 0, 9)
kp_src   = 'primary' if kp_now is not None else 'from_layer4'

# 5) IMF Bz coupling (southward = stronger M-I coupling → GEC influence)
bz_score = norm(-(bz_now or 0), -5, 30) if bz_now is not None else None
bz_src   = 'primary' if bz_now is not None else 'missing'

COMPONENTS = {
    'Generator Potential (from L3)':  {'score': gen_score,   'source': 'derived_from_layer3', 'dynamic': True},
    'Iono. Conductivity (L4)':      {'score': iono_score,  'source': iono_src, 'dynamic': True},
    'GEC Potential Deviation':      {'score': pot_score,   'source': pot_src,  'dynamic': True},
    'Geomag. Disturbance (Kp)':         {'score': kp_score,    'source': kp_src,   'dynamic': True},
    'IMF Bz Coupling':               {'score': bz_score,    'source': bz_src,   'dynamic': True},
}

available   = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable = [k for k, v in COMPONENTS.items() if v['score'] is None]
layer5_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unknown' if layer5_score is None
         else 'quiet'   if layer5_score < 0.3
         else 'moderate' if layer5_score < 0.6
         else 'active')
dominant_l5 = max(available, key=available.get) if available else 'none'

# Kp consistency: separate 1-min raw value from score value
kp_1m_raw   = kp_now          # directly from API (can be 0.0 during live measurement)
kp_for_score = kp_val          # from Layer-0 context or estimated value if needed
kp_score_src = kp_src

W = 68
print('=' * W)
print('LAYER 5 – GLOBAL ELECTRIC CIRCUIT – STATE ASSESSMENT')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s*20) + '░' * (20-int(s*20))
        print(f'  {name:<34} {bar}  {s:.3f}  [{comp["source"]}]')
    else:
        print(f'  {name:<34} {"─"*20}  n/a   [missing]')
print('-' * W)
print(f'  Score:         {layer5_score:.3f}  ({len(available)}/{len(COMPONENTS)} components)')
print(f'  Confidence:    {confidence:.0%}')
print(f'  Level:         {level.upper()}')
print(f'  Dominant:      {dominant_l5}')
print(f'  V_iono:        {V_iono_mod:.1f} kV  (Δ {delta_V_pct:+.1f}% vs. Ref)')
print(f'  I_total:       {I_total_mod:.1f} A   (Δ {delta_I_pct:+.1f}% vs. Ref)')
print('=' * W)

# Radar
cats   = list(available.keys())
vals_r = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_r + [vals_r[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(242,166,35,0.18)',
        line=dict(color='#F2A623', width=2.5), name='Layer 5'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5]*(len(cats)+1), theta=cats+[cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Activity Threshold'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 5 – GEC Activity Profile | Score: {layer5_score:.3f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=12)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0,1])),
        height=450, showlegend=True,
        margin=dict(l=80, r=80, t=70, b=40)
    )
    fig.show()


LAYER 5 – GLOBAL ELECTRIC CIRCUIT – STATE ASSESSMENT
  Generator Potential (from L3)      █████████░░░░░░░░░░░  0.459  [derived_from_layer3]
  Iono. Conductivity (L4)            ████████░░░░░░░░░░░░  0.429  [from_layer4]
  GEC Potential Deviation            █████░░░░░░░░░░░░░░░  0.270  [derived]
  Geomag. Disturbance (Kp)           ██░░░░░░░░░░░░░░░░░░  0.111  [primary]
  IMF Bz Coupling                    ░░░░░░░░░░░░░░░░░░░░  0.037  [primary]
--------------------------------------------------------------------
  Score:         0.261  (5/5 components)
  Confidence:    100%
  Level:         QUIET
  Dominant:      Generator Potential (from L3)
  V_iono:        324.3 kV  (Δ +8.1% vs. Ref)
  I_total:       1081.1 A   (Δ +8.1% vs. Ref)


In [10]:
# ============================================================
# EXPORT – layer5_test_state.json
# ============================================================

# Schumann downstream: GEC state modulates Schumann amplitude
_schumann_l5 = (
    'GEC potential elevated – increased Schumann excitation energy'
    if delta_V_pct > 10
    else 'GEC potential reduced – damped Schumann excitation'
    if delta_V_pct < -10
    else f'GEC near reference (Δ {delta_V_pct:+.1f}%) – normal Schumann conditions'
)

# state_summary
_gen_s  = f'Generator strength {generator_strength:.2f} (CAPE={cape_now:.0f} J/kg).'
_pot_s  = f'V_iono={V_iono_mod:.0f} kV (Δ {delta_V_pct:+.1f}% vs. Carnegie).'
_cur_s  = f'I_total={I_total_mod:.0f} A (Δ {delta_I_pct:+.1f}%).'
_kp_s   = (f'Kp_1m={kp_1m_raw:.1f} (score_Kp={kp_for_score:.1f}).'
           if kp_1m_raw is not None else 'Kp n/a.')
_bz_s   = f'IMF Bz={bz_now:+.1f} nT.' if bz_now is not None else 'IMF Bz n/a.'

state_summary = ' '.join([
    f'Layer-5-State: {level}.', _gen_s, _pot_s, _cur_s, _kp_s, _bz_s,
    f'Data completeness: {confidence:.0%}.'
])

layer5_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 5,
    'name':  'Global Electric Circuit',

    'score':      layer5_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamic components',
    'score_note': 'Solar wind values (SW_speed, SW_density) are context data, not part of the GEC score components.',
    'external_context_missing': [k for k, v in {
        'SW_speed_kms': sw_speed, 'SW_density_cm3': sw_density}.items() if v is None],
    'dominant_component': dominant_l5,
    'missing_components': unavailable,

    'components': {
        k: {
            'score':   round(v['score'], 4) if v['score'] is not None else None,
            'source':  v['source'],
            'dynamic': v['dynamic'],
            **({'note': 'Potential from CAPE/instability; no confirmed active thunderstorm situation'}
               if 'Generator' in k else {})
        }
        for k, v in COMPONENTS.items()
    },

    # The core: modelled GEC state
    'gec_state': {
        'V_ionosphere_kV':      V_iono_mod,
        'I_total_A':            I_total_mod,
        'E_fair_weather_Vm':    E_mod,
        'J_fair_weather_pAm2':  J_mod,
        'generator_strength':   generator_strength,
        'iono_cond_factor':     round(iono_cond_factor, 3),
        'kp_cond_mod':          round(kp_cond_mod, 3),
        'delta_V_pct':          delta_V_pct,
        'delta_I_pct':          delta_I_pct,
    },

    'gec_reference': GEC_REF,

    'coupling_matrix': {
        k: round(v, 4) for k, v in coupling.items()
    },

    'raw_values': {
        'Kp_current_1m':      kp_1m_raw,
        'Kp_used_for_score':  kp_for_score,
        'Kp_source':          kp_score_src,
        'Kp_max_24h':         kp_max_24h,
        'IMF_Bz_nT':     bz_now,
        'SW_speed_kms':  sw_speed,
        'SW_density_cm3':sw_density,
        'xray_class':    xray_class,
        'F10.7_sfu':     f107_now,
        'CAPE_mean_Jkg': cape_now,
    },

    'thresholds': {
        'gec_elevated_pct': 15.0,
        'gec_strong_pct':   30.0,
        'note': 'gec_elevated triggers at +15%, gec_strong at +30%'
    },

    'flags': {
        'gec_elevated':           bool(delta_V_pct > 15.0),
        'gec_suppressed':         bool(delta_V_pct < -15.0),
        'generator_active':       bool(l3_thunder_flag),
        'iono_conductivity_high': bool(iono_cond_factor > 1.1),
        'geomagnetic_storm':      bool(kp_val >= 5),
        'bz_southward_coupled':   bool(bz_now is not None and bz_now <= -5),
    },

    'downstream_expectation': {
        'layer6_schumann':    _schumann_l5,
        'schumann_amplitude': (
            'elevated amplitude expected'
            if delta_V_pct > 10
            else 'damped amplitude possible'
            if delta_V_pct < -10
            else 'normal amplitude'
        ),
        'layer7_engine': (
            f'GEC-Score {layer5_score:.3f} ({level}) – '
            f'dominant: {dominant_l5}'
        ),
    },

    'layer_context': {
        'L0': {'score': layer0['score'], 'level': layer0['level']} if layer0 else None,
        'L1': {'score': layer1['score'], 'level': layer1['level'],
               'ocean_conductivity': l1_leitf} if layer1 else None,
        'L2': {'score': layer2['score'], 'level': layer2['level']} if layer2 else None,
        'L3': {'score': layer3['score'], 'level': layer3['level'],
               'thunder_score': l3_thunder_score,
               'CAPE_mean': l3_cape,
               'thunder_trigger': l3_thunder_flag} if layer3 else None,
        'L4': {'score': layer4['score'], 'level': layer4['level'],
               'cavity_height_km': l4_cavity_h,
               'ideal_cavity_mode1_Hz': l4_sr1_hz,
               'real_SR1_reference_Hz': 7.83,
               'ioniz_score': l4_ioniz} if layer4 else None,
    },

    'state_summary': state_summary,
}

# numpy-Typen bereinigen
def _to_python(obj):
    import numpy as np
    if isinstance(obj, dict):  return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
layer5_state = _to_python(layer5_state)

with open('layer5_test_state.json', 'w', encoding='utf-8') as f:
    json.dump(layer5_state, f, indent=2, ensure_ascii=False)

print('layer5_test_state.json saved')
print(json.dumps(layer5_state, indent=2, ensure_ascii=False))


layer5_test_state.json saved
{
  "timestamp": "2026-05-14T07:36:33.623056Z",
  "layer": 5,
  "name": "Global Electric Circuit",
  "score": 0.2612,
  "level": "quiet",
  "confidence": 1.0,
  "score_basis": "5/5 dynamic components",
  "score_note": "Solar wind values (SW_speed, SW_density) are context data, not part of the GEC score components.",
  "external_context_missing": [
    "SW_speed_kms",
    "SW_density_cm3"
  ],
  "dominant_component": "Generator Potential (from L3)",
  "missing_components": [],
  "components": {
    "Generator Potential (from L3)": {
      "score": 0.4594,
      "source": "derived_from_layer3",
      "dynamic": true,
      "note": "Potential from CAPE/instability; no confirmed active thunderstorm situation"
    },
    "Iono. Conductivity (L4)": {
      "score": 0.4287,
      "source": "from_layer4",
      "dynamic": true
    },
    "GEC Potential Deviation": {
      "score": 0.27,
      "source": "derived",
      "dynamic": true
    },
    "Geomag. Disturbanc

---
## Summary Layer 5

| Aspekt | Inhalt |
|--------|--------|
| **Role** | Electrical circulation system – system architecture of Schumann resonance |
| **Generator** | Layer 3 (thunderstorm, CAPE) → Ionosphere charging |
| **Upper electrode** | Layer 4 (Ionosphere, F10.7 conductivity) |
| **Lower electrode** | Layer 1 (oceans 3.2 S/m, 71% area) |
| **Model** | Carnegie reference × generator factor × conductivity factor × Kp-mod |
| **→ Layer 6** | GEC potential level → Schumann amplitude + frequency conditions |
| **→ Layer 7** | `gec_state` + `coupling_matrix` as engine input |
| **Output** | `layer5_test_state.json` with `gec_state`, `coupling_matrix`, full L0–L4 context |

> **Nächster Schritt:** `layer6_Resonance_Field.ipynb` – Resonance Field / Schumann